In [1]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from sqlalchemy import create_engine, text
import os
import pathlib

BASE_DIR = pathlib.Path(os.getcwd())
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

DB_PATH = BASE_DIR / 'data' / 'banco.db'
engine  = create_engine(f'sqlite:///{DB_PATH}', echo=False)

df_c   = pd.read_sql('SELECT * FROM tb_clientes', engine)
df_c   = df_c.drop(columns=['cluster_id','cluster_nome'], errors='ignore')
df_cl  = pd.read_sql('SELECT id_cliente, cluster_id, cluster_nome FROM tb_clusters', engine)
df_rfm = pd.read_sql('SELECT * FROM tb_rfm', engine)

df = df_c.merge(df_cl,  on='id_cliente', how='left')
df = df.merge(df_rfm,   on='id_cliente', how='left')

CORES_PERFIL = {
    'Primeiros Passos':    '3498db',
    'Trajetória Crescente':'2ecc71',
    'Potencial Oculto':    'f39c12',
    'Self Made':           '9b59b6',
    'Old Money':           'e74c3c',
}

print(f"Banco:   {DB_PATH}")
print(f"Clientes: {len(df):,} | Colunas: {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")

Banco:   c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO\data\banco.db
Clientes: 300,000 | Colunas: 52
Missing values: 0


In [2]:
DEPARA = [
    {
        'cluster_id':      0,
        'cluster_nome':    'Primeiros Passos',
        'renda_min':       800,
        'renda_max':       2500,
        'score_min':       0,
        'score_max':       500,
        'idade_min':       18,
        'idade_max':       26,
        'perfil_risco':    'Alto',
        'canal_ataque':    'App / Digital',
        'produto_primario':'Cartão de Crédito',
        'produto_secundario':'CDB Digital',
        'meta_conversao':  '10%',
        'setores':         'Serviços, Logística, Indústria',
        'observacao':      'Jovem iniciando carreira. Foco em inclusão financeira e primeiro produto.',
    },
    {
        'cluster_id':      1,
        'cluster_nome':    'Trajetória Crescente',
        'renda_min':       2500,
        'renda_max':       8000,
        'score_min':       400,
        'score_max':       650,
        'idade_min':       25,
        'idade_max':       45,
        'perfil_risco':    'Médio',
        'canal_ataque':    'Gerente / App',
        'produto_primario':'Financiamento',
        'produto_secundario':'Seguro de Vida',
        'meta_conversao':  '20%',
        'setores':         'Financeiro, Educação, Comercial, Engenharia',
        'observacao':      'Relacionamento longo, produtos clássicos. Potencial de cross-sell.',
    },
    {
        'cluster_id':      2,
        'cluster_nome':    'Potencial Oculto',
        'renda_min':       6000,
        'renda_max':       20000,
        'score_min':       450,
        'score_max':       750,
        'idade_min':       28,
        'idade_max':       50,
        'perfil_risco':    'Baixo',
        'canal_ataque':    'Gerente / Digital',
        'produto_primario':'Investimento',
        'produto_secundario':'Previdência Privada',
        'meta_conversao':  '25%',
        'setores':         'Tecnologia, Saúde, Comunicação',
        'observacao':      'Ganha bem mas pouco explorado. Alta receptividade a investimentos.',
    },
    {
        'cluster_id':      3,
        'cluster_nome':    'Self Made',
        'renda_min':       15000,
        'renda_max':       60000,
        'score_min':       600,
        'score_max':       900,
        'idade_min':       30,
        'idade_max':       55,
        'perfil_risco':    'Baixo',
        'canal_ataque':    'Gerente Exclusivo',
        'produto_primario':'Crédito Imobiliário',
        'produto_secundario':'Câmbio Internacional',
        'meta_conversao':  '30%',
        'setores':         'Gestão, Jurídico, Tecnologia',
        'observacao':      'Alta renda, produtos parciais. Foco em imobiliário e câmbio.',
    },
    {
        'cluster_id':      4,
        'cluster_nome':    'Old Money',
        'renda_min':       50000,
        'renda_max':       120000,
        'score_min':       750,
        'score_max':       1000,
        'idade_min':       40,
        'idade_max':       70,
        'perfil_risco':    'Muito Baixo',
        'canal_ataque':    'Private Banking',
        'produto_primario':'Cartão Black',
        'produto_secundario':'Gestão de Patrimônio',
        'meta_conversao':  '40%',
        'setores':         'Alta Renda',
        'observacao':      'Carteira consolidada. Foco exclusivo em upgrades e produtos premium.',
    },
]

df_depara = pd.DataFrame(DEPARA)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'De-Para Segmentação'

ws.merge_cells('A1:N1')
ws['A1'].value = 'De-Para de Segmentação — Motor de Recomendação'
ws['A1'].font = Font(bold=True, size=13, color='FFFFFF')
ws['A1'].fill = PatternFill(start_color='1a1a2e', end_color='1a1a2e', fill_type='solid')
ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 30

ws.merge_cells('A2:N2')
ws['A2'].value = 'Versão 1.0 — Gerado automaticamente | Fonte: K-Means + PNAD Contínua 2024'
ws['A2'].font = Font(italic=True, size=10, color='555555')
ws['A2'].alignment = Alignment(horizontal='center')
ws.row_dimensions[2].height = 18

headers = [
    'ID Cluster','Segmento','Renda Mín (R$)','Renda Máx (R$)',
    'Score Mín','Score Máx','Idade Mín','Idade Máx',
    'Perfil Risco','Canal de Ataque','Produto Primário',
    'Produto Secundário','Meta Conversão','Observação'
]
for col_idx, h in enumerate(headers, 1):
    cell = ws.cell(row=3, column=col_idx, value=h)
    cell.font = Font(bold=True, color='FFFFFF', size=10)
    cell.fill = PatternFill(start_color='EC0000', end_color='EC0000', fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
ws.row_dimensions[3].height = 35

for i, row in df_depara.iterrows():
    linha  = i + 4
    cor    = CORES_PERFIL[row['cluster_nome']]
    valores = [
        row['cluster_id'], row['cluster_nome'],
        row['renda_min'], row['renda_max'],
        row['score_min'], row['score_max'],
        row['idade_min'], row['idade_max'],
        row['perfil_risco'], row['canal_ataque'],
        row['produto_primario'], row['produto_secundario'],
        row['meta_conversao'], row['observacao'],
    ]
    for col_idx, val in enumerate(valores, 1):
        cell = ws.cell(row=linha, column=col_idx, value=val)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        if col_idx == 2:
            cell.fill = PatternFill(start_color=cor, end_color=cor, fill_type='solid')
            cell.font = Font(bold=True, color='FFFFFF')
        elif i % 2 == 1:
            cell.fill = PatternFill(start_color='F5F5F5', end_color='F5F5F5', fill_type='solid')
    ws.row_dimensions[linha].height = 35

larguras = [10,22,14,14,10,10,10,10,14,18,22,22,14,45]
for col_idx, w in enumerate(larguras, 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = w
ws.freeze_panes = 'A4'

ws2 = wb.create_sheet('Histórico de Versões')
ws2.merge_cells('A1:E1')
ws2['A1'].value = 'Histórico de Versões'
ws2['A1'].font = Font(bold=True, size=12, color='FFFFFF')
ws2['A1'].fill = PatternFill(start_color='1a1a2e', end_color='1a1a2e', fill_type='solid')
ws2['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 28

for col_idx, h in enumerate(['Versão','Data','Responsável','Alteração','Motivo'], 1):
    cell = ws2.cell(row=2, column=col_idx, value=h)
    cell.font = Font(bold=True, color='FFFFFF', size=10)
    cell.fill = PatternFill(start_color='EC0000', end_color='EC0000', fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')

historico = [
    ('v1.0','01/01/2025','Calebe Bizarro','Criação inicial','Primeiro treinamento K-Means'),
    ('v1.1','01/02/2025','Calebe Bizarro','Ajuste faixas de renda','Novos dados de mercado'),
    ('v1.2','01/03/2025','Calebe Bizarro','Adição canal digital','Crescimento uso do app'),
]
for i, linha in enumerate(historico):
    for col_idx, val in enumerate(linha, 1):
        cell = ws2.cell(row=i+3, column=col_idx, value=val)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if i % 2 == 1:
            cell.fill = PatternFill(start_color='F5F5F5', end_color='F5F5F5', fill_type='solid')

for col_idx, w in enumerate([10,14,20,35,35], 1):
    ws2.column_dimensions[get_column_letter(col_idx)].width = w

caminho_excel = str(BASE_DIR / 'docs' / 'depara_segmentacao.xlsx')
wb.save(caminho_excel)
print(f"Excel salvo: {caminho_excel}")

Excel salvo: c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO\docs\depara_segmentacao.xlsx


In [3]:
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_depara"))
    conn.commit()

df_depara.to_sql('tb_depara', engine, if_exists='replace', index=False)

print("=== tb_depara no SQLite ===")
with engine.connect() as conn:
    for row in conn.execute(text("""
        SELECT cluster_nome, canal_ataque, produto_primario, meta_conversao
        FROM tb_depara
    """)):
        print(f"  {row[0]:22} | {row[1]:18} | {row[2]:22} | meta {row[3]}")

=== tb_depara no SQLite ===
  Primeiros Passos       | App / Digital      | Cartão de Crédito      | meta 10%
  Trajetória Crescente   | Gerente / App      | Financiamento          | meta 20%
  Potencial Oculto       | Gerente / Digital  | Investimento           | meta 25%
  Self Made              | Gerente Exclusivo  | Crédito Imobiliário    | meta 30%
  Old Money              | Private Banking    | Cartão Black           | meta 40%


In [4]:
def classificar_por_depara(row):
    renda  = row['renda_mensal']
    score  = row['score_credito']
    idade  = row['idade']

    if renda >= 50000 and score >= 750:
        return 4, 'Old Money'
    if renda >= 15000 and score >= 600:
        return 3, 'Self Made'
    if renda >= 6000 and score >= 450 and idade >= 28:
        return 2, 'Potencial Oculto'
    if renda >= 2500 and idade >= 25:
        return 1, 'Trajetória Crescente'
    return 0, 'Primeiros Passos'

df['cluster_depara']      = df.apply(lambda r: classificar_por_depara(r)[0], axis=1)
df['cluster_nome_depara'] = df.apply(lambda r: classificar_por_depara(r)[1], axis=1)
df['concordancia']        = (df['cluster_id'] == df['cluster_depara']).astype(int)

taxa = df['concordancia'].mean()
print(f"Taxa de concordância K-Means vs De-Para: {taxa:.1%}")

print(f"\nDiscordâncias por cluster (top 10):")
disc = df[df['concordancia']==0].groupby(
    ['cluster_nome','cluster_nome_depara']
).size().reset_index(name='qtd').sort_values('qtd', ascending=False).head(10)
print(disc.to_string(index=False))

Taxa de concordância K-Means vs De-Para: 40.7%

Discordâncias por cluster (top 10):
        cluster_nome  cluster_nome_depara   qtd
           Self Made     Potencial Oculto 57515
Trajetória Crescente     Potencial Oculto 49872
    Potencial Oculto Trajetória Crescente 41322
    Primeiros Passos Trajetória Crescente 10312
           Old Money            Self Made  9156
           Self Made Trajetória Crescente  4990
    Primeiros Passos     Potencial Oculto  3810
           Old Money     Potencial Oculto   785
           Old Money Trajetória Crescente    87
    Potencial Oculto            Self Made    39


In [5]:
df_depara_result = df[[
    'id_cliente','cluster_depara','cluster_nome_depara','concordancia'
]].copy()

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_depara_clientes"))
    conn.commit()

df_depara_result.to_sql('tb_depara_clientes', engine, if_exists='replace', index=False)

print("=== DIAGNÓSTICO FINAL 05 ===\n")
print(f"Taxa de concordância: {df['concordancia'].mean():.1%}")

print(f"\nTabelas no banco:")
with engine.connect() as conn:
    for row in conn.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")):
        count = conn.execute(text(f"SELECT COUNT(*) FROM {row[0]}")).scalar()
        print(f"  {row[0]:25} | {count:,} registros")

=== DIAGNÓSTICO FINAL 05 ===

Taxa de concordância: 40.7%

Tabelas no banco:
  tb_clientes               | 300,000 registros
  tb_clusters               | 300,000 registros
  tb_depara                 | 5 registros
  tb_depara_clientes        | 300,000 registros
  tb_estados_cidades        | 64 registros
  tb_ibge_salarios          | 69 registros
  tb_perfil_clusters        | 5 registros
  tb_propensao              | 300,000 registros
  tb_rfm                    | 300,000 registros
